# py-methyl-toolkit — End-to-end analysis demo

**Data:** 6 Bismark `.cov.gz` files — 3 CONTROL vs 3 CD55 samples (2 donors, up to 2 replicates each)

| Sample | Group | Donor | Rep |
|---|---|---|---|
| GSM9212480 CONTROLDONOR3 REP1 | control | D3 | R1 |
| GSM9212482 CD55DONOR3 REP1 | cd55 | D3 | R1 |
| GSM9212484 CONTROLDONOR4 REP1 | control | D4 | R1 |
| GSM9212486 CD55DONOR4 REP1 | cd55 | D4 | R1 |
| GSM9212488 CONTROLDONOR3 REP2 | control | D3 | R2 |
| GSM9212490 CD55DONOR3 REP2 | cd55 | D3 | R2 |

**Pipeline:**
1. Load → 2. Build AnnData → 3. QC → 4. Filter → 5. Unite → 6. Diff meth → 7. DMRs

## 1 · Imports & configuration

In [ ]:
from pathlib import Path
import gc
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

# epykit modules
from epykit.io.bismark import read_bismark_coverage
from epykit.io.anndata_builder import build_anndata, save, load
from epykit.core.methyldata import MethylData
from epykit.stats.tests import calculate_diff_meth
from epykit.stats.dmr import merge_dmrs
from epykit.plot import qc as qc_plots

import logging
logging.basicConfig(level=logging.INFO)


# Directories
DATA_DIR   = Path(".")                 # folder containing this notebook
RESULT_DIR = Path("results")
PLOT_DIR   = Path("plots")
RESULT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

print("epykit imported ✓")
print(f"Results → {RESULT_DIR.resolve()}")
print(f"Plots   → {PLOT_DIR.resolve()}")

pymethyl imported ✓
Results → D:\Coding\Projeler\methyl_lib\test\results
Plots   → D:\Coding\Projeler\methyl_lib\test\plots


## 2 · Load Bismark coverage files

In [2]:
# Each entry: (file_path, sample_id, group, donor, replicate)
SAMPLES = [
    (DATA_DIR / "GSM9212480_CONTROLDONOR3_REP1_1_val_1_bismark_bt2_pe.deduplicated.bismark.cov.gz",
     "CTRL_D3_R1", "control", "D3", "R1"),
    (DATA_DIR / "GSM9212482_CD55DONOR3_REP1_1_val_1_bismark_bt2_pe.deduplicated.bismark.cov.gz",
     "CD55_D3_R1", "cd55",    "D3", "R1"),
    (DATA_DIR / "GSM9212484_CONTROLDONOR4_REP1_1_val_1_bismark_bt2_pe.deduplicated.bismark.cov.gz",
     "CTRL_D4_R1", "control", "D4", "R1"),
    (DATA_DIR / "GSM9212486_CD55DONOR4_REP1_1_val_1_bismark_bt2_pe.deduplicated.bismark.cov.gz",
     "CD55_D4_R1", "cd55",    "D4", "R1"),
    (DATA_DIR / "GSM9212488_CONTROLDONOR3_REP2_1_val_1_bismark_bt2_pe.deduplicated.bismark.cov.gz",
     "CTRL_D3_R2", "control", "D3", "R2"),
    (DATA_DIR / "GSM9212490_CD55DONOR3_REP2_1_val_1_bismark_bt2_pe.deduplicated.bismark.cov.gz",
     "CD55_D3_R2", "cd55",    "D3", "R2"),
]

dfs = {}
for path, sid, group, donor, rep in SAMPLES:
    df = read_bismark_coverage(path, min_coverage=10)
    dfs[sid] = df
    print(f"{sid:12s}  {len(df):>9,} sites   {df['coverage'].median():.0f}× median cov")

print("\nExample — first 5 rows of CTRL_D3_R1:")
dfs["CTRL_D3_R1"].head()

CTRL_D3_R1    18,889,020 sites   12× median cov
CD55_D3_R1    26,293,555 sites   13× median cov
CTRL_D4_R1    20,647,415 sites   12× median cov
CD55_D4_R1    29,033,024 sites   14× median cov
CTRL_D3_R2    23,363,403 sites   13× median cov
CD55_D3_R2    21,637,418 sites   12× median cov

Example — first 5 rows of CTRL_D3_R1:


chr,start,end,beta,methylated,unmethylated,coverage
str,i64,i64,f64,i32,i32,i32
"""chr1""",13283,13283,100.0,10,0,10
"""chr1""",13284,13284,100.0,14,0,14
"""chr1""",13302,13302,100.0,10,0,10
"""chr1""",13303,13303,92.857143,13,1,14
"""chr1""",13418,13418,0.0,0,11,11


## 3 · Build AnnData (samples × sites)

In [ ]:
from epykit.io import build_anndata_chunked

# Build per-sample observation metadata
obs_meta = pd.DataFrame(
    [
        {"sample_id": sid, "group": group, "donor": donor, "replicate": rep}
        for _, sid, group, donor, rep in SAMPLES
    ]
).set_index("sample_id")

# Define sample order to match SAMPLES and obs_meta
sample_ids = [sid for _, sid, _, _, _ in SAMPLES]

# Build list of per-sample DataFrames in that order
sample_dfs = [dfs[sid] for sid in sample_ids]
del dfs  # free memory
gc.collect()

# Optional sanity check: metadata index matches sample_ids
assert set(obs_meta.index) == set(sample_ids)

# Build the AnnData — union of all sites (NaN where a site isn't covered)
# Option 1: Sparse beta matrix (recommended)
adata = build_anndata_chunked(
    sample_ids=sample_ids,
    dataframes=sample_dfs,
    obs_metadata=obs_meta,
    join_type="outer",
    chunk_size=1_000_000,  # process 1M sites at a time
    zarr_path="results/temp_adata.zarr",
)


print(adata)
print("\nSample metadata:")
adata.obs


INFO:pymethyl.io.anndata_builder_chunked:Building AnnData (chunked) from 6 samples (join_type='outer', chunk_size=1000000) …
INFO:pymethyl.io.anndata_builder_chunked:  Step 1/5: computed int64 locus IDs for all samples
INFO:pymethyl.io.anndata_builder_chunked:  Step 2/5: global locus index built (42245510 loci)


AttributeError: module 'zarr' has no attribute 'DirectoryStore'

## 4 · MethylData wrapper

In [ ]:
mdata = MethylData(adata)
print(mdata)
print(f"\nSamples : {mdata.n_samples}")
print(f"Sites   : {mdata.n_sites:,}")

## 5 · QC — coverage statistics

In [ ]:
cov_stats = mdata.coverage_stats()
print("Coverage statistics per sample:")
cov_stats

In [ ]:
fig = qc_plots.coverage_hist(mdata, max_cov=100)
fig.savefig(PLOT_DIR / "01_coverage_hist.png", bbox_inches="tight")
plt.show()

## 6 · Filter coverage

Keep sites with ≥ **1×** coverage in every sample (a sensible starting point; raise to 5–10 for stricter analyses).

In [ ]:
MIN_COV = 1
MAX_COV = 1000   # remove extreme outliers (PCR artefacts)

mdata_filt = mdata.filter_coverage(min_coverage=MIN_COV, max_coverage=MAX_COV)
print(f"Before filter : {mdata.n_sites:>10,} sites")
print(f"After  filter : {mdata_filt.n_sites:>10,} sites")
print(f"Retained      : {mdata_filt.n_sites / mdata.n_sites * 100:.1f} %")

## 7 · Subset to CpG context

> If the `.cov.gz` files don't have a context column the toolkit will still work — all sites are treated as CpG by default.

In [ ]:
if "context" in mdata_filt.sites.columns:
    mdata_cpg = mdata_filt.subset_context("CpG")
    print(f"CpG sites : {mdata_cpg.n_sites:,}")
else:
    mdata_cpg = mdata_filt
    print("No 'context' column — using all sites as CpG")
    print(f"Sites : {mdata_cpg.n_sites:,}")

## 8 · Unite — keep only sites covered in ALL samples

In [ ]:
mdata_united = mdata_cpg.unite(how="intersect")
print(f"Sites covered in ALL {mdata_united.n_samples} samples: {mdata_united.n_sites:,}")

## 9 · QC — global methylation per sample

In [ ]:
global_meth = mdata_united.global_methylation()
print("Global methylation (% per sample):")
global_meth

In [ ]:
fig = qc_plots.methylation_distribution(mdata_united)
fig.savefig(PLOT_DIR / "02_meth_distribution.png", bbox_inches="tight")
plt.show()

## 10 · QC — PCA (coloured by group)

In [ ]:
fig = qc_plots.pca(mdata_united, color_by="group")
fig.savefig(PLOT_DIR / "03_pca.png", bbox_inches="tight")
plt.show()

## 11 · QC — sample-to-sample correlation heatmap

In [ ]:
fig = qc_plots.sample_correlation(mdata_united)
fig.savefig(PLOT_DIR / "04_sample_corr.png", bbox_inches="tight")
plt.show()

## 12 · Differential methylation — CONTROL vs CD55

Uses Logistic GLM + LRT (with HC0 overdispersion correction) because we have 3 replicates per group.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

results = calculate_diff_meth(
    mdata_united,
    treatment_col="group",
    test="auto",           # auto = GLM (3 replicates/group)
    fdr_method="BH",
    verbose=True,
)

print(f"\nTotal sites tested  : {len(results):,}")
print(f"Significant (q<0.05): {(results.qvalue < 0.05).sum():,}")
print(f"Significant (q<0.01): {(results.qvalue < 0.01).sum():,}")
results.head(10)

## 13 · Top differentially methylated CpGs (DMCs)

In [ ]:
# Significant DMCs (q < 0.05, |mean_diff| > 10%)
dmcs = results[(results.qvalue < 0.05) & (results.mean_diff.abs() > 10)].copy()
dmcs = dmcs.sort_values("mean_diff", key=abs, ascending=False)

print(f"DMCs (q<0.05 & |Δβ|>10%) : {len(dmcs):,}")
dmcs[["chr", "start", "end", "pvalue", "qvalue", "mean_diff"]].head(20)

In [ ]:
# Simple volcano-style scatter
import numpy as np

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["red" if (q < 0.05 and abs(d) > 10) else "steelblue"
          for q, d in zip(results.qvalue, results.mean_diff)]
ax.scatter(results.mean_diff, -np.log10(results.pvalue + 1e-300),
           c=colors, s=2, alpha=0.4, rasterized=True)
ax.axvline(-10, color="grey", lw=0.8, ls="--")
ax.axvline(+10, color="grey", lw=0.8, ls="--")
ax.axhline(-np.log10(0.05), color="grey", lw=0.8, ls=":")
ax.set_xlabel("Mean methylation difference (CD55 − CONTROL, %)")
ax.set_ylabel("-log₁₀(p-value)")
ax.set_title("Volcano plot: CD55 vs CONTROL")
fig.tight_layout()
fig.savefig(PLOT_DIR / "05_volcano.png", dpi=150, bbox_inches="tight")
plt.show()

## 14 · Merge DMCs → Differentially Methylated Regions (DMRs)

In [ ]:
dmrs = merge_dmrs(
    results,
    qvalue_col="qvalue",
    qvalue_cutoff=0.05,
    diff_col="mean_diff",
    min_sites=3,          # at least 3 CpGs per DMR
)

print(f"DMRs found : {len(dmrs):,}")
if len(dmrs):
    print("\nTop 10 DMRs by number of CpGs:")
    dmrs.sort_values("n_sites", ascending=False).head(10)

In [ ]:
if len(dmrs):
    # DMR direction breakdown
    if "direction" in dmrs.columns:
        print(dmrs.direction.value_counts().to_string())
    dmrs.head()

## 15 · Save results

In [ ]:
# Save AnnData (can be loaded instantly for future work)
save(mdata_united.adata, RESULT_DIR / "methyldata_united.h5ad")
print("Saved methyldata_united.h5ad")

# Save differential methylation results
results.to_csv(RESULT_DIR / "diff_meth.tsv", sep="\t", index=False)
print("Saved diff_meth.tsv")

# Save significant DMCs only
dmcs.to_csv(RESULT_DIR / "dmcs_q05_d10.tsv", sep="\t", index=False)
print("Saved dmcs_q05_d10.tsv")

# Save DMRs
if len(dmrs):
    dmrs.to_csv(RESULT_DIR / "dmrs.tsv", sep="\t", index=False)
    print("Saved dmrs.tsv")

print("\n✓ All results saved to", RESULT_DIR.resolve())

## 16 · (Optional) Reload saved data

In a future session you can skip the loading/processing steps and jump straight to analysis:

In [ ]:
# mdata_reloaded = MethylData(load(RESULT_DIR / "methyldata_united.h5ad"))
# print(mdata_reloaded)